# **Import Library**

In [4]:
!pip install pyspark
!pip install findspark
import os
import pandas as pd

## Google Drive bağlanma

In [5]:
from google.colab import drive
drive.mount ('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# **Data Processing**

In [7]:
findspark.init()

NameError: ignored

In [8]:
spark = SparkSession.builder.appName("Air Quality Project").getOrCreate()

NameError: ignored

In [ ]:
#Veriyi okuma
dataframe = spark.read.csv("/content/drive/MyDrive/Air-Quality-Merged-Data/Clinton.csv", header= True, inferSchema= True)

In [ ]:
#Verinin ilk 5 satırını görüntüleme
dataframe.show()

In [ ]:
#Verinin kolon ve veri tiplerini gösterme
dataframe.printSchema()

In [ ]:
#Verinin Temel İstatistiklerini Görüntüleme
dataframe = dataframe.withColumnRenamed("PM2.5", "PM2_5")
dataframe.describe().show()

# **Özellik Mühendisliği**

In [ ]:
#Grab Columns Name

def grab_col_names_spark(dataframe, cat_th=10, car_th=20):
    """

    This function takes a PySpark DataFrame and returns the names of categorical, numerical,
    and cardinal categorical variables.

    Parameters
    ------
        dataframe: PySpark DataFrame
                The dataframe for which variable names are desired.
        cat_th: int, optional
                Threshold for numerical but categorical variables
        car_th: int, optinal
                Threshold for categorical but cardinal variables

    Returns
    ------
        cat_cols: list
                List of categorical variables
        num_cols: list
                List of numerical variables
        cat_but_car: list
                List of cardinal categorical variables

    """

    # Get column data types
    col_data_types = dataframe.dtypes

    # Categorical columns
    cat_cols = [col_name for col_name, dtype in col_data_types if dtype == "string"]

    # Numerical columns which look like categorical
    num_but_cat = [col_name for col_name, dtype in col_data_types if dataframe.select(countDistinct(col_name)).collect()[0][0] < cat_th and dtype != "string"]

    # Categorical columns which are actually cardinal
    cat_but_car = [col_name for col_name in cat_cols if dataframe.select(countDistinct(col_name)).collect()[0][0] > car_th]

    # Update cat_cols and num_cols
    cat_cols = cat_cols + num_but_cat
    cat_cols = [col_name for col_name in cat_cols if col_name not in cat_but_car]
    num_cols = [col_name for col_name, dtype in col_data_types if dtype != "string"]
    num_cols = [col_name for col_name in num_cols if col_name not in num_but_cat]

    print(f"Observations: {dataframe.count()}")
    print(f"Variables: {len(dataframe.columns)}")
    print(f'cat_cols: {len(cat_cols)}')
    print(f'num_cols: {len(num_cols)}')
    print(f'cat_but_car: {len(cat_but_car)}')
    print(f'num_but_cat: {len(num_but_cat)}')
    return cat_cols, num_cols, cat_but_car

# To test the function, you would use:
# cat_cols, num_cols, cat_but_car = grab_col_names_spark(spark_dataframe)  # Assuming spark_dataframe is your PySpark dataframe

# Note: For now, the function is provided. Actual testing requires a PySpark DataFrame.
cat_cols, num_cols, cat_but_car = grab_col_names_spark(dataframe)



In [ ]:
# Aykırı Değer Sorgulama

def outlier_thresholds_spark(dataframe, col_name, q1=0.25, q3=0.75):

    num_cols_filtered = [col for col in num_cols if col != "DateTime"]

    # Calculate quantiles
    quantiles = dataframe.approxQuantile(col_name, [q1, q3], 0.05)
    quartile1, quartile3 = quantiles[0], quantiles[1]

    # Calculate IQR
    interquantile_range = quartile3 - quartile1

    # Calculate outlier thresholds
    up_limit = quartile3 + 1.5 * interquantile_range
    low_limit = quartile1 - 1.5 * interquantile_range

    return low_limit, up_limit

def check_outlier_spark(dataframe, col_name):
    # Get outlier thresholds
    low_limit, up_limit = outlier_thresholds_spark(dataframe, col_name)

    # Check for outliers
    result_df = dataframe.filter((col(col_name) > up_limit) | (col(col_name) < low_limit))

    # Return boolean based on presence of outliers
    return result_df.count() > 0


# for col in num_cols_spark:
#     print(col, check_outlier_spark(spark_dataframe, col))

outlier_thresholds_spark, check_outlier_spark


In [ ]:
# Baskılama Yöntemi

def replace_with_thresholds_spark(dataframe, num_cols: list) -> DataFrame:
    # DateTime sütununu num_cols listesinden çıkar
    if "DateTime" in num_cols:
        num_cols.remove("DateTime")

    for col_name in num_cols:
        low_limit, up_limit = outlier_thresholds_spark(dataframe, col_name)
        dataframe = dataframe.withColumn(col_name,
                                         when(col(col_name) < low_limit, low_limit).otherwise(
                                         when(col(col_name) > up_limit, up_limit).otherwise(col(col_name))))
        print(f"Processed column: {col_name}")

    return dataframe


dataframe = replace_with_thresholds_spark(dataframe, num_cols)
dataframe.show()


In [ ]:
# Eksik Değer Sorgulama Tablosu

def missing_values_table_spark(dataframe, na_name=False):

    total_records = dataframe.count()

    # Eksik değerleri sorgulama
    missing_counts = dataframe.select([count(when(col(c).isNull(), c)).alias(c) for c in dataframe.columns])

    # Eksik değer oranlarını hesaplama
    missing_ratios = dataframe.select([(count(when(col(c).isNull(), c)) / total_records).alias(c) for c in dataframe.columns])

    # Sonuçları yazdırma
    print("Missing Values:")
    missing_counts.show()

    print("Missing Ratios:")
    missing_ratios.show()

    if na_name:
        na_columns = [c for c in dataframe.columns if dataframe.where(dataframe[c].isNull()).count() > 0]
        return na_columns

missing_values_table_spark(dataframe)


In [ ]:
# Eksik Değer Sorgulama Tablosu

def missing_data_table_spark(dataframe):
    total_records = dataframe.count()

    # Eksik değer sayılarını sorgulama
    missing_counts = dataframe.select([count(when(col(c).isNull(), c)).alias(c) for c in dataframe.columns]).collect()[0]

    # Eksik değer oranlarını hesaplama
    missing_ratios = dataframe.select([(count(when(col(c).isNull(), c)) / total_records).alias(c) for c in dataframe.columns]).collect()[0]

    # Sonuçları DataFrame haline getirme
    result = [(c, missing_counts[c], missing_ratios[c]) for c in dataframe.columns]
    result_df = spark.createDataFrame(result, ["Column_Name", "Total_Missing", "Missing_Ratio"])

    return result_df.orderBy(result_df.Total_Missing.desc())


missing_df = missing_data_table_spark(dataframe)
missing_df.show()


In [ ]:
def fill_missing_with_mean_spark(dataframe, num_cols: list):
    """
    Fills missing values in numeric columns with the mean of that column.

    Parameters:
    - dataframe: PySpark DataFrame with missing values
    - num_cols: List of numeric column names

    Returns:
    - DataFrame with missing values filled
    """
    for col_name in num_cols:
        mean_value = dataframe.select(mean(col(col_name))).collect()[0][0]
        dataframe = dataframe.na.fill(mean_value, subset=[col_name])

    return dataframe


dataframe = fill_missing_with_mean_spark(dataframe, num_cols)
missing_data_table_spark(dataframe).show()


In [ ]:
for column in num_cols:
    dataframe = dataframe.withColumn(column, round(dataframe[column], 3))

dataframe.show()


In [ ]:
# PySpark DataFrame'ini Pandas DataFrame'ine dönüştürme
pandas_df = dataframe.toPandas()

# Pandas DataFrame'ini CSV dosyası olarak kaydetme
pandas_df.to_csv("/content/drive/MyDrive/Air-Quality-Merged-Data/a.csv", index=False)


## Not: Aşağıda buradaki tüm dosyaları bu fonksiyonlar ile çalıştırmak için toplu yazılmış kod bulunmaktadır!!!

In [ ]:
input_directory = "/content/drive/MyDrive/Air-Quality-Merged-Data"
output_directory = "/content/drive/MyDrive/Air-Quality-Merged-Data-new"

for filename in os.listdir(input_directory):
  if filename.endswith(".csv"):
    filepath = os.path.join(input_directory, filename)

    dataframe = spark.read.csv(filepath, header= True, inferSchema= True)
"""
    dataframe = grab_col_names_spark(dataframe)
    dataframe = outlier_thresholds_spark
    dataframe = check_outlier_spark
    dataframe = replace_with_thresholds_spark(dataframe, num_cols)
    dataframe = missing_values_table_spark(dataframe)
    dataframe = fill_missing_with_mean_spark(dataframe)

    output_filepath = os.path.join(output_directory,"Pro" + filename)
    dataframe.toPandas().to_csv(output_filepath, index= False)

print("Done!")
"""

## Yukarıda tek tek yapılan işlemlerin topluca verilerimize yapılmış hali


In [9]:
from pyspark.sql import SparkSession
import findspark
from pyspark.sql import DataFrame
from pyspark.sql.functions import *
from pyspark.sql import functions
from pyspark.sql import DataFrameStatFunctions as statFunc


import os
import pandas as pd

findspark.init()

spark = SparkSession.builder.appName("Air Quality Project").getOrCreate()

input_directory = "/content/drive/MyDrive/Air-Quality-Merged-Data"
output_directory = "/content/drive/MyDrive/Air-Quality-Merged-Data-new"


for filename in os.listdir(input_directory):
  if filename.endswith(".csv"):
    filepath = os.path.join(input_directory, filename)

    dataframe = spark.read.csv(filepath, header= True, inferSchema= True)
    dataframe = dataframe.withColumnRenamed("PM2.5", "PM2_5")


    #Grab Columns Name

    def grab_col_names_spark(dataframe, cat_th=10, car_th=20):
        """

        This function takes a PySpark DataFrame and returns the names of categorical, numerical,
        and cardinal categorical variables.

        Parameters
        ------
            dataframe: PySpark DataFrame
                    The dataframe for which variable names are desired.
            cat_th: int, optional
                    Threshold for numerical but categorical variables
            car_th: int, optinal
                    Threshold for categorical but cardinal variables

        Returns
        ------
            cat_cols: list
                    List of categorical variables
            num_cols: list
                    List of numerical variables
            cat_but_car: list
                    List of cardinal categorical variables

        """

        # Get column data types
        col_data_types = dataframe.dtypes

        # Categorical columns
        cat_cols = [col_name for col_name, dtype in col_data_types if dtype == "string"]

        # Numerical columns which look like categorical
        num_but_cat = [col_name for col_name, dtype in col_data_types if dataframe.select(countDistinct(col_name)).collect()[0][0] < cat_th and dtype != "string"]

        # Categorical columns which are actually cardinal
        cat_but_car = [col_name for col_name in cat_cols if dataframe.select(countDistinct(col_name)).collect()[0][0] > car_th]

        # Update cat_cols and num_cols
        cat_cols = cat_cols + num_but_cat
        cat_cols = [col_name for col_name in cat_cols if col_name not in cat_but_car]
        num_cols = [col_name for col_name, dtype in col_data_types if dtype != "string"]
        num_cols = [col_name for col_name in num_cols if col_name not in num_but_cat]

        print(f"Observations: {dataframe.count()}")
        print(f"Variables: {len(dataframe.columns)}")
        print(f'cat_cols: {len(cat_cols)}')
        print(f'num_cols: {len(num_cols)}')
        print(f'cat_but_car: {len(cat_but_car)}')
        print(f'num_but_cat: {len(num_but_cat)}')
        return cat_cols, num_cols, cat_but_car

    # To test the function, you would use:
    # cat_cols, num_cols, cat_but_car = grab_col_names_spark(spark_dataframe)  # Assuming spark_dataframe is your PySpark dataframe

    # Note: For now, the function is provided. Actual testing requires a PySpark DataFrame.
    cat_cols, num_cols, cat_but_car = grab_col_names_spark(dataframe)

    # Aykırı Değer Sorgulama

    def outlier_thresholds_spark(dataframe, col_name, q1=0.25, q3=0.75):

        num_cols_filtered = [col for col in num_cols if col != "DateTime"]

        # Calculate quantiles
        quantiles = dataframe.approxQuantile(col_name, [q1, q3], 0.05)
        quartile1, quartile3 = quantiles[0], quantiles[1]

        # Calculate IQR
        interquantile_range = quartile3 - quartile1

        # Calculate outlier thresholds
        up_limit = quartile3 + 1.5 * interquantile_range
        low_limit = quartile1 - 1.5 * interquantile_range

        return low_limit, up_limit

    def check_outlier_spark(dataframe, col_name):
        # Get outlier thresholds
        low_limit, up_limit = outlier_thresholds_spark(dataframe, col_name)

        # Check for outliers
        result_df = dataframe.filter((col(col_name) > up_limit) | (col(col_name) < low_limit))

        # Return boolean based on presence of outliers
        return result_df.count() > 0


    # for col in num_cols_spark:
    #     print(col, check_outlier_spark(spark_dataframe, col))

    outlier_thresholds_spark, check_outlier_spark

    # Baskılama Yöntemi

    def replace_with_thresholds_spark(dataframe, num_cols: list) -> DataFrame:
        # DateTime sütununu num_cols listesinden çıkar
        if "DateTime" in num_cols:
            num_cols.remove("DateTime")

        for col_name in num_cols:
            low_limit, up_limit = outlier_thresholds_spark(dataframe, col_name)
            dataframe = dataframe.withColumn(col_name,
                                            when(col(col_name) < low_limit, low_limit).otherwise(
                                            when(col(col_name) > up_limit, up_limit).otherwise(col(col_name))))
            print(f"Processed column: {col_name}")

        return dataframe


    dataframe = replace_with_thresholds_spark(dataframe, num_cols)
    dataframe.show()

    # Eksik Değer Sorgulama Tablosu

    def missing_values_table_spark(dataframe, na_name=False):

        total_records = dataframe.count()

        # Eksik değerleri sorgulama
        missing_counts = dataframe.select([count(when(col(c).isNull(), c)).alias(c) for c in dataframe.columns])

        # Eksik değer oranlarını hesaplama
        missing_ratios = dataframe.select([(count(when(col(c).isNull(), c)) / total_records).alias(c) for c in dataframe.columns])

        # Sonuçları yazdırma
        print("Missing Values:")
        missing_counts.show()

        print("Missing Ratios:")
        missing_ratios.show()

        if na_name:
            na_columns = [c for c in dataframe.columns if dataframe.where(dataframe[c].isNull()).count() > 0]
            return na_columns

    missing_values_table_spark(dataframe)

    # Eksik Değer Sorgulama Tablosu

    def missing_data_table_spark(dataframe):
        total_records = dataframe.count()

        # Eksik değer sayılarını sorgulama
        missing_counts = dataframe.select([count(when(col(c).isNull(), c)).alias(c) for c in dataframe.columns]).collect()[0]

        # Eksik değer oranlarını hesaplama
        missing_ratios = dataframe.select([(count(when(col(c).isNull(), c)) / total_records).alias(c) for c in dataframe.columns]).collect()[0]

        # Sonuçları DataFrame haline getirme
        result = [(c, missing_counts[c], missing_ratios[c]) for c in dataframe.columns]
        result_df = spark.createDataFrame(result, ["Column_Name", "Total_Missing", "Missing_Ratio"])

        return result_df.orderBy(result_df.Total_Missing.desc())


    missing_df = missing_data_table_spark(dataframe)
    missing_df.show()


    def fill_missing_with_mean_spark(dataframe, num_cols: list):
        """
        Fills missing values in numeric columns with the mean of that column.

        Parameters:
        - dataframe: PySpark DataFrame with missing values
        - num_cols: List of numeric column names

        Returns:
        - DataFrame with missing values filled
        """
        for col_name in num_cols:
            mean_value = dataframe.select(mean(col(col_name))).collect()[0][0]
            dataframe = dataframe.na.fill(mean_value, subset=[col_name])

        return dataframe


    dataframe = fill_missing_with_mean_spark(dataframe, num_cols)
    missing_data_table_spark(dataframe).show()

    output_filepath = os.path.join(output_directory,"Pro" + filename)
    dataframe.toPandas().to_csv(output_filepath, index= False)









print("Done!")





FileNotFoundError: ignored

## Verimizde bulunan virgüllü sayıları yuvarlama kodu

In [ ]:
# Lists to store processed and unprocessed file names
processed_files = []
unprocessed_files = []

def round_numeric_columns(df, csv_file):
    modified = False  # Flag to check if the DataFrame was modified
    for column_name in df.columns:
        if df[column_name].dtype in ('float64', 'float32'):
            df[column_name] = df[column_name].round(2)
            modified = True
    if modified:
        processed_files.append(csv_file)
    else:
        unprocessed_files.append(csv_file)
    return df

input_directory = "/content/drive/MyDrive/Air-Quality-Merged-Data-new"
output_directory = "/content/drive/MyDrive/Rounded"

# Same CSV processing code
for csv_file in csv_files:
    # Read the CSV file
    file_path = os.path.join(input_directory, csv_file)
    df = pd.read_csv(file_path)

    # Round the numeric columns
    df_rounded = round_numeric_columns(df, csv_file)

    # Save the results to the specified folder
    output_path = os.path.join(output_directory, "Rounded_" + csv_file)
    df_rounded.to_csv(output_path, index=False)

processed_files, unprocessed_files


# **Model Kodları**

### Gerekli kütüphaneler

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
!pip install fbprophet

In [ ]:
!pip install cmdstanpy==1.0.4
!pip install prophet --upgrade


## Veri setimizde bulunan "DateTime", "PM10", "PM2_5" sütunlarını göre veri setimizi tekrar oluşturduk.

In [ ]:
# Tüm dosyaların bulunduğu klasör yolu
folder_path = "/content/drive/MyDrive/Rounded"

# Dosyaların listesini alın
all_files = os.listdir(folder_path)

# Tüm veriyi depolamak için boş bir DataFrame oluşturun
all_data = pd.DataFrame(columns=["DateTime", "PM10", "PM2_5"])

# Tüm dosyaları döngüde okuyun ve ana DataFrame'e ekleyin
for file in all_files:
    file_path = os.path.join(folder_path, file)
    try:
        data = pd.read_csv(file_path, usecols=["DateTime", "PM10", "PM2_5"])
        all_data = all_data.append(data)
    except:
        print(f"{file} dosyasında ilgili sütunlar bulunamadı.")


output_path = "/content/drive/MyDrive/Rounded/combined_data.csv"

# Son olarak birleştirilmiş veriyi kaydedin
all_data.to_csv(output_path, index=False)


## PM2_5 için Prophet kütüphanesi kullanarak modelleme

In [ ]:
from prophet import Prophet
# Verinizi yükleyin
data = pd.read_csv("/content/drive/MyDrive/Rounded/combined_data.csv", parse_dates=['DateTime'])

# Veriyi günlük bazda yeniden örnekleyin
daily_data = data.resample('Y', on='DateTime').mean().reset_index()
daily_data.dropna(inplace=True)

# Veriyi %80 eğitim ve %20 test olarak bölme
train_size = int(0.8 * len(daily_data))
train_data_pm25 = daily_data.iloc[:train_size][['DateTime', 'PM2_5']].rename(columns={'DateTime': 'ds', 'PM2_5': 'y'})
test_data_pm25 = daily_data.iloc[train_size:][['DateTime', 'PM2_5']].rename(columns={'DateTime': 'ds', 'PM2_5': 'y'})

# Prophet modelini başlatın ve eğitin
model_pm25 = Prophet(yearly_seasonality=True, weekly_seasonality=True, daily_seasonality=False)
model_pm25.fit(train_data_pm25)

# Test seti süresi için tahmin yapma
future_pm25 = model_pm25.make_future_dataframe(periods=10, freq='Y')
forecast_pm25 = model_pm25.predict(future_pm25)
# Setting the style for seaborn
sns.set_style("whitegrid")

# Plotting the actual and predicted values
plt.figure(figsize=(15, 7))
plt.plot(daily_data['DateTime'], daily_data['PM2_5'], label='Actual PM2.5', color='blue')
plt.plot(forecast_pm25['ds'], forecast_pm25['yhat'], label='Predicted PM2.5', color='red', linestyle='--')
plt.fill_between(forecast_pm25['ds'], forecast_pm25['yhat_lower'], forecast_pm25['yhat_upper'], color='red', alpha=0.2)

# Adding titles and labels
plt.title("Actual vs Predicted PM2.5 Levels", fontsize=16)
plt.xlabel("Date", fontsize=14)
plt.ylabel("PM2.5 Levels", fontsize=14)
plt.legend()
plt.tight_layout()

plt.show()

## PM10 için Prophet kütüphanesi kullanarak modelleme

In [ ]:
import pandas as pd
from prophet import Prophet
import matplotlib.pyplot as plt

# Veriyi yükleyin
data = pd.read_csv("/content/drive/MyDrive/Rounded/combined_data.csv", parse_dates=['DateTime'])

# Veriyi yıllık bazda yeniden örnekleyin
daily_data = data.resample('Y', on='DateTime').mean().reset_index()
daily_data.dropna(inplace=True)

# Veriyi %80 eğitim ve %20 test olarak bölme
train_size = int(0.8 * len(daily_data))

# PM10 için eğitim ve test verisini hazırlama
train_data_pm10 = daily_data.iloc[:train_size][['DateTime', 'PM10']].rename(columns={'DateTime': 'ds', 'PM10': 'y'})
test_data_pm10 = daily_data.iloc[train_size:][['DateTime', 'PM10']].rename(columns={'DateTime': 'ds', 'PM10': 'y'})

# PM10 için Prophet modelini başlatma ve eğitme
model_pm10 = Prophet(yearly_seasonality=True, weekly_seasonality=True, daily_seasonality=False)
model_pm10.fit(train_data_pm10)

# PM10 için test seti süresince tahmin yapma
future_pm10 = model_pm10.make_future_dataframe(periods=10, freq='Y')
forecast_pm10 = model_pm10.predict(future_pm10)

# PM10 için gerçek ve tahmin edilen değerleri grafikleştirme
plt.figure(figsize=(15, 7))
plt.plot(daily_data['DateTime'], daily_data['PM10'], label='Actual PM10', color='blue')
plt.plot(forecast_pm10['ds'], forecast_pm10['yhat'], label='Predicted PM10', color='green', linestyle='--')
plt.fill_between(forecast_pm10['ds'], forecast_pm10['yhat_lower'], forecast_pm10['yhat_upper'], color='green', alpha=0.2)

# PM10 grafiği için başlık ve etiket ekleme
plt.title("Actual vs Predicted PM10 Levels", fontsize=16)
plt.xlabel("Date", fontsize=14)
plt.ylabel("PM10 Levels", fontsize=14)
plt.legend()
plt.tight_layout()

plt.show()


## Hata metriklerini hesaplama

In [ ]:
# PM2.5 için hata metriklerini hesaplama
actual_pm25 = daily_data['PM2_5'].iloc[train_size:].values
predicted_pm25 = forecast_pm25['yhat'].iloc[-len(test_data_pm25):].values

mae_pm25 = mean_absolute_error(actual_pm25, predicted_pm25)
rmse_pm25 = np.sqrt(mean_squared_error(actual_pm25, predicted_pm25))

# PM10 için hata metriklerini hesaplama
actual_pm10 = daily_data['PM10'].iloc[train_size:].values
predicted_pm10 = forecast_pm10['yhat'].iloc[-len(test_data_pm10):].values

mae_pm10 = mean_absolute_error(actual_pm10, predicted_pm10)
rmse_pm10 = np.sqrt(mean_squared_error(actual_pm10, predicted_pm10))

print(f"PM2.5 için MAE: {mae_pm25:.2f}")
print(f"PM2.5 için RMSE: {rmse_pm25:.2f}")
print(f"PM10 için MAE: {mae_pm10:.2f}")
print(f"PM10 için RMSE: {rmse_pm10:.2f}")
